# Notebook 3: Node Classification — Full Pipeline

Now we apply everything to a real benchmark: the **Cora** citation network. This is the de-facto baseline dataset for node classification GNNs.

**Prerequisites:** Notebooks 1–2

**What you'll learn:**
- Cora dataset: structure, features, splits
- Full GCN training pipeline with validation
- Transductive learning and why it differs from standard supervised ML
- Regularization for graphs (dropout, weight decay)
- Visualizing embeddings with t-SNE
- Ablation studies: depth, width, learning rate

**By the end:** Beat 80% accuracy on Cora's test set using a GCN you tune yourself.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
import torch_geometric.transforms as T

torch.manual_seed(42)
np.random.seed(42)

---
## 1. The Cora Dataset

**Cora** is a citation network of scientific papers:
- **Nodes:** 2,708 papers
- **Edges:** 5,429 citation links (undirected)
- **Node features:** 1,433-dim bag-of-words vector (word presence in paper)
- **Classes:** 7 paper topics (e.g., Neural Networks, Reinforcement Learning...)
- **Train/Val/Test split:** 140 / 500 / 1000 nodes

### Transductive vs. Inductive Learning

In standard supervised ML, test samples are *unseen*. In **transductive** learning (Cora), the model *sees all nodes and all edges during training* — it just doesn't know the labels for val/test nodes. This matters because:
- GNNs aggregate neighbor information, so test nodes can leverage their (labeled) neighbors
- The model can't generalize to entirely new graphs (that's *inductive* learning — Notebook 5)

This is why GNNs on Cora can achieve high accuracy with only 140 labeled training nodes!

In [ ]:
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]

print('=== Cora Dataset ===')
print(f'Nodes:          {data.num_nodes:>6}')
print(f'Edges:          {data.num_edges:>6}')
print(f'Node features:  {data.num_node_features:>6}')
print(f'Classes:        {dataset.num_classes:>6}')
print(f'Train nodes:    {data.train_mask.sum().item():>6}')
print(f'Val nodes:      {data.val_mask.sum().item():>6}')
print(f'Test nodes:     {data.test_mask.sum().item():>6}')
print()
print(f'Avg degree:     {data.num_edges / data.num_nodes:.2f}')
print()

# Label distribution in train set
from collections import Counter
train_labels = data.y[data.train_mask].tolist()
print('Train label distribution:', Counter(train_labels))

In [ ]:
# Visualize class distribution
class_names = ['Case_Based', 'Genetic_Algorithms', 'Neural_Networks',
               'Probabilistic_Methods', 'Reinforcement_Learning',
               'Rule_Learning', 'Theory']

counts = [(data.y == i).sum().item() for i in range(7)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Full label distribution
axes[0].bar(class_names, counts, color=plt.cm.tab10(np.arange(7)))
axes[0].set_xticklabels(class_names, rotation=35, ha='right', fontsize=9)
axes[0].set_title('All nodes — class distribution')
axes[0].set_ylabel('Count')

# Feature sparsity
sparsity = (data.x == 0).float().mean().item()
axes[1].hist(data.x.sum(dim=1).numpy(), bins=50, color='steelblue', edgecolor='white')
axes[1].set_title(f'Node feature sum distribution (sparsity: {sparsity:.1%})')
axes[1].set_xlabel('Number of non-zero features')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

---
## 2. The GCN Model

In [ ]:
class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GCNConv(in_channels, hidden_channels, cached=True)
        self.conv2 = GCNConv(hidden_channels, out_channels, cached=True)
    
    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x
    
    @torch.no_grad()
    def get_embeddings(self, x, edge_index):
        self.eval()
        x = F.relu(self.conv1(x, edge_index))
        return x  # intermediate embeddings


model = GCN(
    in_channels=dataset.num_features,
    hidden_channels=64,
    out_channels=dataset.num_classes,
    dropout=0.5
)
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

---
## 3. Training Loop with Validation

In [ ]:
def train(model, data, optimizer):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate(model, data):
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    
    results = {}
    for split in ['train', 'val', 'test']:
        mask = getattr(data, f'{split}_mask')
        correct = (pred[mask] == data.y[mask]).sum().item()
        results[split] = correct / mask.sum().item()
    return results


torch.manual_seed(42)
model = GCN(dataset.num_features, 64, dataset.num_classes, dropout=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

history = {'loss': [], 'train_acc': [], 'val_acc': [], 'test_acc': []}
best_val_acc = 0
best_model_state = None

for epoch in range(1, 201):
    loss = train(model, data, optimizer)
    accs = evaluate(model, data)
    
    history['loss'].append(loss)
    history['train_acc'].append(accs['train'])
    history['val_acc'].append(accs['val'])
    history['test_acc'].append(accs['test'])
    
    # Save best model by val accuracy
    if accs['val'] > best_val_acc:
        best_val_acc = accs['val']
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
    
    if epoch % 50 == 0 or epoch == 1:
        print(f'Epoch {epoch:>3} | Loss: {loss:.4f} | '
              f'Train: {accs["train"]:.4f} | Val: {accs["val"]:.4f} | Test: {accs["test"]:.4f}')

print(f'\nBest val acc: {best_val_acc:.4f}')
model.load_state_dict(best_model_state)
final_accs = evaluate(model, data)
print(f'Final test acc (best model): {final_accs["test"]:.4f}')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history['loss'], color='red', label='Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-entropy loss')
axes[0].set_title('Training Loss')
axes[0].legend()

axes[1].plot(history['train_acc'], label='Train', color='steelblue')
axes[1].plot(history['val_acc'], label='Val', color='orange')
axes[1].plot(history['test_acc'], label='Test', color='green', linestyle='--')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 4. Visualizing Learned Embeddings with t-SNE

t-SNE projects high-dimensional embeddings into 2D while preserving local structure. If the GCN learned good representations, nodes of the same class should cluster together.

In [ ]:
model.eval()
embeddings = model.get_embeddings(data.x, data.edge_index).numpy()
labels = data.y.numpy()

print(f'Embedding shape: {embeddings.shape}')
print('Running t-SNE (may take ~30s)...')

tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
embeddings_2d = tsne.fit_transform(embeddings)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# t-SNE colored by class
colors = plt.cm.tab10(np.arange(7))
for cls in range(7):
    mask = labels == cls
    axes[0].scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
                   c=[colors[cls]], label=class_names[cls], s=10, alpha=0.7)
axes[0].set_title('t-SNE of GCN embeddings (colored by class)')
axes[0].legend(markerscale=3, bbox_to_anchor=(1.0, 1.0))
axes[0].set_xlabel('t-SNE dim 1')
axes[0].set_ylabel('t-SNE dim 2')

# Highlight train/val/test split
split_colors = np.array(['lightgray'] * data.num_nodes, dtype=object)
split_colors[data.train_mask.numpy()] = 'red'
split_colors[data.val_mask.numpy()] = 'orange'
split_colors[data.test_mask.numpy()] = 'green'

axes[1].scatter(embeddings_2d[:, 0], embeddings_2d[:, 1],
               c=split_colors, s=10, alpha=0.6)
from matplotlib.patches import Patch
leg2 = [Patch(color='red', label='Train (140)'),
        Patch(color='orange', label='Val (500)'),
        Patch(color='green', label='Test (1000)'),
        Patch(color='lightgray', label='Unlabeled')]
axes[1].legend(handles=leg2)
axes[1].set_title('t-SNE — train/val/test split')

plt.tight_layout()
plt.show()

---
## 5. Ablation Study — Impact of Depth and Width

A key insight for GNNs: more layers ≠ better. Unlike CNNs, deeper GNNs can suffer from **over-smoothing** — node embeddings become too similar as they average over larger and larger neighborhoods.

In [ ]:
class GCN_Deep(nn.Module):
    """GCN with variable number of layers."""
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(in_channels, hidden_channels))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
        self.convs.append(GCNConv(hidden_channels, out_channels))
    
    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs[:-1]):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.convs[-1](x, edge_index)
        return x


results_by_depth = {}
for num_layers in [1, 2, 3, 4, 6, 8]:
    torch.manual_seed(42)
    m = GCN_Deep(dataset.num_features, 64, dataset.num_classes, num_layers)
    opt = torch.optim.Adam(m.parameters(), lr=0.01, weight_decay=5e-4)
    
    best_val = 0
    best_test = 0
    for epoch in range(200):
        m.train(); opt.zero_grad()
        out = m(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward(); opt.step()
        
        m.eval()
        with torch.no_grad():
            pred = m(data.x, data.edge_index).argmax(1)
        val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
        test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
        if val_acc > best_val:
            best_val = val_acc
            best_test = test_acc
    
    results_by_depth[num_layers] = (best_val, best_test)
    print(f'Layers: {num_layers} | Best Val: {best_val:.4f} | Best Test: {best_test:.4f}')

fig, ax = plt.subplots(figsize=(7, 4))
depths = list(results_by_depth.keys())
val_accs = [results_by_depth[d][0] for d in depths]
test_accs = [results_by_depth[d][1] for d in depths]
ax.plot(depths, val_accs, 'o-', label='Val acc', color='orange')
ax.plot(depths, test_accs, 's-', label='Test acc', color='green')
ax.set_xlabel('Number of GCN layers')
ax.set_ylabel('Accuracy')
ax.set_title('Over-smoothing: accuracy vs. depth')
ax.legend()
ax.set_xticks(depths)
plt.tight_layout()
plt.show()

---
# MINI-PROJECT 3: Beat 82% on Cora

**Task:** Tune a GCN (or extend it) to beat 82% test accuracy on Cora. The original paper achieves ~81.5% with a 2-layer GCN. Can you do better?

**What to try:**
1. Hyperparameter search: hidden channels (16, 32, 64, 128), learning rate (0.001–0.05), dropout (0.3–0.7)
2. Add a 3rd layer with a skip connection (residual GCN)
3. Try different optimizers (Adam vs SGD with momentum)
4. Add batch normalization between layers

**Requirements:**
- Report test accuracy for at least 3 different configurations
- Plot validation accuracy curves for each configuration
- Write a short explanation (in a markdown cell) of what worked and what didn't

**IMPORTANT:** Always pick your model using *validation* accuracy, then report test accuracy once. Never tune on test!

In [ ]:
# MINI-PROJECT SKELETON

class ImprovedGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, 
                 dropout=0.5, use_batchnorm=False):
        super().__init__()
        self.dropout = dropout
        self.use_batchnorm = use_batchnorm
        
        # TODO: define layers
        # Hint: try adding nn.BatchNorm1d(hidden_channels) between conv layers
        pass
    
    def forward(self, x, edge_index):
        # TODO: forward pass
        pass


def run_experiment(config: dict, data, dataset, epochs=300):
    """Run one training experiment and return val/test accuracy history."""
    torch.manual_seed(42)
    
    # TODO: instantiate model from config dict
    # config keys: hidden_channels, dropout, lr, weight_decay, use_batchnorm
    
    model = None   # replace with your model
    optimizer = None   # replace with optimizer
    
    val_history = []
    best_val = 0
    best_test = 0
    
    for epoch in range(epochs):
        # TODO: train one epoch
        # TODO: evaluate and track val/test accuracy
        pass
    
    return val_history, best_val, best_test


# TODO: define 3+ configurations and run experiments
configs = [
    {'hidden_channels': 64, 'dropout': 0.5, 'lr': 0.01, 'weight_decay': 5e-4, 'use_batchnorm': False},
    # Add more configs here...
]

results = []
for cfg in configs:
    val_hist, best_val, best_test = run_experiment(cfg, data, dataset)
    results.append((cfg, best_val, best_test))
    print(f'Config: {cfg} -> Val: {best_val:.4f}, Test: {best_test:.4f}')

# TODO: plot validation curves for each config
# TODO: print summary table of results

### Hints

<details>
<summary>Hint 1 — Residual GCN layer</summary>

```python
class ResGCNConv(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = GCNConv(channels, channels)
        self.bn = nn.BatchNorm1d(channels)
    
    def forward(self, x, edge_index):
        return F.relu(self.bn(self.conv(x, edge_index)) + x)  # skip connection
```
</details>

<details>
<summary>Hint 2 — Quick hyperparameter grid</summary>

```python
import itertools
hidden_sizes = [64, 128]
dropouts = [0.4, 0.5, 0.6]
lrs = [0.005, 0.01]
for h, d, lr in itertools.product(hidden_sizes, dropouts, lrs):
    cfg = {'hidden_channels': h, 'dropout': d, 'lr': lr, 'weight_decay': 5e-4}
    ...
```
</details>

---
## What's Next?

In **Notebook 4**, we attack the key weakness of GCN: it treats all neighbors equally. **Graph Attention Networks (GAT)** let each node *learn which neighbors matter most* — and we'll visualize those attention weights.